# 集群混合并行训练 · 导学：多卡训练与运行验证

在进入正式实验之前，先明确三件事：**做什么、怎么做、怎么验收**。本 notebook 不启动训练，用于**核对环境、过一遍检查清单、建立全局认知**。

## 本节你将学到

- 实验的三个目标和验收标准
- ModelArts 平台的完整操作路径
- 启动训练前的 8 项检查清单
- 怎么判断训练是否正常
- 实验报告交什么

## 实验环境

- **平台**：华为云 ModelArts + JupyterLab
- **硬件**：Ascend 910B NPU，单卡 64GB HBM2e，单机 4 卡
- **模型**：Qwen2.5-7B（通义千问 70 亿参数）

> 本节是导学课——只做环境检查和认知建立，**不启动真实训练**。训练在下一个 notebook（实操篇《MindSpeed 训练配置与启动流程》）里做。

---

## 一、做什么：实验目标

集群混合并行训练实验是分布式训练的配套实验。学完《大模型多机多卡训练原理与并行策略》《AllReduce、HCCL 与通信性能分析》《MindSpeed 分布式训练流程》三门理论课后，你在这里把它们串起来——**亲手在 NPU 上跑一次多卡训练**。

三个具体目标：

| 目标 | 说明 | 验收 |
|------|------|------|
| **① 启动训练** | TP/PP/DP 配置正确，训练不报错 | 看到 loss 正常打印 |
| **② 分析日志** | 从日志提取三个关键指标 | loss / 吞吐 / 通信占比 |
| **③ 完成报告** | 配置记录 + 日志分析 + 反思 | 算/跑/看/想四部分齐全 |

---

## 二、怎么做：ModelArts 平台

本实验在**华为云 ModelArts 平台**上进行，不用 SSH 登录集群。下面按截图逐步走一遍创建流程。

### 第 1 步：登录华为云控制台，切换区域到西南-贵阳一

昇腾 910B（snt9b 系列）资源只在华为云少数区域提供，本课程统一使用西南-贵阳一（实测资源可用）。如果创建时找不到对应规格，先检查区域是否选对。

![控制台区域选择](images/modelarts-step1-console.png)

### 第 2 步：进入 ModelArts，创建 Notebook

左侧菜单：**开发环境 → Notebook → 创建**。

![创建 Notebook](images/modelarts-step2-create.png)

### 第 3 步：选择镜像

工作环境选公共镜像 **mindspeed_llm_2.2.0**（AI 引擎选 MindSpeed-LLM）。镜像预装 CANN、torch_npu、MindSpeed-LLM，环境免搭建。

![选择镜像](images/modelarts-step3-image.png)

### 第 4 步：选择规格

规格选 **4×Ascend 910B（单卡 64GB）**；存储配置选 EVS，容量 50GB 以上（权重约 15GB + 转换产物 + 数据）；建议开启自动停止（如 4 小时），防止忘记关机。

![选择规格](images/modelarts-step4-flavor.png)

### 第 5 步：实例运行后，打开 JupyterLab

实例状态变为"运行中"后，点击"接入环境"进入 JupyterLab。NPU 资源紧张时创建可能排队或失败，换个时段重试即可。
![打开 JupyterLab](images/modelarts-step5-jupyterlab.png)

### 第 6 步：认清 JupyterLab 的分工

- **notebook**：参数计算、配置生成、日志解析
- **终端**：跑 torchrun 多卡训练

![JupyterLab 与终端](images/modelarts-step6-terminal.png)

所有实验文件放在 `/home/ma-user/work/` 目录——这是持久化目录，停止实例后不丢失。

---

## 三、启动前检查：8 项清单

这 8 项任何一项不满足，训练都无法正常启动或很快中断。逐项过一遍，每项都标注了在实操 notebook 的哪一步验证：

| # | 检查项 | 怎么确认 | 在哪验证 |
|---|--------|---------|---------|
| ① | CANN 环境已加载 | `npu-smi info` 有输出，能看到 4 张卡 | 实操 notebook 的 Step 1 |
| ② | HCCL 通信正常 | AllReduce 校验通过，每个 rank 打印 OK | 实操 notebook 的 Step 1 |
| ③ | 数据产物存在 | `alpaca_text_document.bin/.idx` 两个文件都在 | 实操 notebook 的 Step 3 |
| ④ | 权重转换产物存在 | `mp_rank_*` 目录共 4 个（= TP×PP = 2×2） | 实操 notebook 的 Step 2 |
| ⑤ | TP×PP ≤ 总卡数 | 2×2 = 4 ≤ 4，成立 | 全局配置 |
| ⑥ | 显存预算通过 | 见下方估算 | 全局配置 |
| ⑦ | MASTER_ADDR / MASTER_PORT 正确 | 单机用 localhost + 未占用端口 | 实操 notebook 的 Step 4 |
| ⑧ | 输出目录可写 | 训练脚本能创建输出目录并写日志 | 实操 notebook 的 Step 4 |

**第⑥项的显存估算**：Qwen2.5-7B 总参数约 7.6B，按 TP=2 PP=2 切分后每卡约 1.9B 参数。bf16 训练加 AdamW 优化器，每参数约 20 字节，模型部分约 38GB；加上激活值（seq=8192、开了重计算，占比很小）和运行开销，**每卡约 42GB / 64GB，满足显存要求**。如果显存不够，三个方向：减小 seq_length、增大 TP 或 PP、开启重计算。

---

## 四、真实环境检查

清单是纸面核对，现在看真实的 NPU。下面两个 cell 在 ModelArts 昇腾实例上运行。

> 如果在普通电脑上运行会报错——你需要先按上面的步骤创建昇腾实例。
>
> **创建实例后先核对**：`npu-smi info` 应显示 4 张卡、单卡显存 64GB。如果卡数或显存不对，停下检查规格是否选错，再继续往下做。

In [1]:
# 查看 NPU 硬件信息
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.1                   Version: 25.5.1                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 1     910B2               | OK            | 98.3        49                0    / 0             |
| 0                         | 0000:01:00.0  | 0           0    / 0          3405 / 65536         |
+===========================+===============+====================================================+
| 2     910B2               | OK            | 102.5       49                0    / 0             |
| 0       

你会看到一个表格，列出每张卡的编号、型号、显存占用、温度。确认两件事：**4 张卡都在**、**显存基本空闲**（训练前应接近 0）。

In [2]:
# 确认 torch_npu 可用
import warnings
warnings.filterwarnings('ignore')
import torch
import torch_npu
print('可用 NPU 数量：', torch.npu.device_count())   # 应输出 4

可用 NPU 数量： 4


---

## 五、运行验证：怎么判断训练正常

训练启动后，关注三个信号。实操 notebook 里你会在日志里找到它们。

### 正常的三个信号

| 信号 | 正常表现 | 怎么看 |
|------|----------|-------|
| **lm loss** | 持续下降，不震荡 | 日志每步打印，如 `lm loss: 8.45` |
| **吞吐量** | tokens/s/p 稳定 | 不忽高忽低 |
| **单步耗时** | 各步基本一致 | 没有某张卡突然变慢 |

### 三大常见异常

| 异常 | 现象 | 首选排查方向 |
|------|------|-------------|
| **OOM** | 显存不足报错 | 减小 seq / 增大 TP·PP / 开重计算 |
| **HCCL timeout** | 通信超时 | 查网络 / 调大 HCCL_CONNECT_TIMEOUT |
| **loss NaN** | loss 突然变 NaN | 降学习率 / 查梯度裁剪 |

> 分布式训练首次启动出现报错属于常见情况，关键是根据报错类型确定排查方向。

---

## 六、HCCL Test：通信验证（走读）

多卡训练的命脉是**通信**——卡间通信不通，训练会立即失败。HCCL（Huawei Collective Communication Library）是昇腾的集合通信库，相当于 GPU 生态中的 NCCL。

这步在实操 notebook 上机时执行（需要多卡）。验证思路很简单：**每张卡放一个数，做一次 AllReduce 求和，校验结果是否正确**。

```python
# 核心代码如下（实操 notebook 会完整执行）
import torch, torch_npu, torch.distributed as dist
dist.init_process_group(backend='hccl')          # 初始化 HCCL 通信
t = torch.ones(1024, 1024, device='npu:0') * 1   # 每张卡放一个值
dist.all_reduce(t, op=dist.ReduceOp.SUM)         # 全局求和
# 4 张卡各放 1、2、3、4 时，求和结果应为 1+2+3+4=10
```

用 `torchrun` 拉起多进程执行：

```bash
source /usr/local/Ascend/ascend-toolkit/set_env.sh 2>/dev/null
export HCCL_WHITELIST_DISABLE=1          # 关闭白名单（单机常用）
export HCCL_CONNECT_TIMEOUT=7200         # 建链超时 7200 秒
torchrun --nproc_per_node 4 hccl_check.py
```

| 环节 | 作用 |
|------|------|
| `backend='hccl'` | 用昇腾的 HCCL 做通信（不是 nccl） |
| `torchrun --nproc_per_node 4` | 拉起 4 个进程，每张卡一个 |
| `HCCL_WHITELIST_DISABLE=1` | 关闭通信白名单，防卡死 |
| `HCCL_CONNECT_TIMEOUT=7200` | 建链超时调大，防 init 卡住 |

> 期望结果：每个 rank 打印 `AllReduce correct = OK`，说明卡间通信完全正常。
>
> 为什么不用 CANN 自带的 `all_reduce_test` 工具？因为 ModelArts 镜像没预装它（需要编译），用 torch_npu 的 `all_reduce` 效果等价，且一定可用。

---

## 七、提交要求

实验报告分四部分——**算、跑、看、想**：

| 部分 | 内容 | 数据来源 |
|------|------|----------|
| **算** | 并行配置记录（TP/PP/DP + GAS 计算过程） | 实操 notebook 的全局配置与配置解读 |
| **跑** | 训练日志摘录（前 10 步 + 最后 10 步） | 实操 notebook 的 run_log |
| **看** | 性能数据（tokens/s/p + 单步耗时 + 通信占比） | 实操 notebook 的日志解析 |
| **想** | 结论反思（为什么选这个配置？能否更好？） | 你的分析 |

评分维度：

```
配置正确性 40% + 日志完整性 30% + 分析深度 30%
```

> **评分重点**：跑通是基础，能否解释「为什么这样配」「有没有更好的配置」决定得分上限。

---

## 小结

本节做了上机前的全部准备：
1. 明确了三个实验目标
2. 走通了 ModelArts 的创建路径（4×Ascend 910B 64GB + mindspeed_llm_2.2.0 镜像）
3. 过了一遍启动前 8 项检查清单（并行配置 TP=2 PP=2 DP=1，每卡约 42GB / 64GB）
4. 了解了正常/异常信号和排查方向
5. 明确了提交要求

下一步进入**实操 notebook**（MindSpeed 训练配置与启动流程）：在 ModelArts 上完成五步闭环——

```
下载权重 → 权重转换 → 数据预处理 → torchrun 启动 → 日志解析
```